In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Dataset Class** -> blueprint of how pytorch's dataset will look like.
It defines :
* __init__() -> how data should be loaded
* __len__() -> returns the total number of samples
* __getitem__(index) -> returns the label and the data at the given index


**DataLoader Class** wraps Dataset and handles batching ,shuffling and parallel loading. 
Data loader control flow:
* At the start of each epoch ,it shuffles indices using a sampler
* it divides the indices into chinks of batch_size
* for each index in the chunk, data samples are fetched from the datset object
* the samples are then collected and combined into a batch via (collate_fn)
* the batch is returned to the main training loop

In [2]:
from sklearn.datasets import make_classification
import torch

In [3]:
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=42     # For reproducibility
)

In [4]:
X , X.shape

(array([[ 1.06833894, -0.97007347],
        [-1.14021544, -0.83879234],
        [-2.8953973 ,  1.97686236],
        [-0.72063436, -0.96059253],
        [-1.96287438, -0.99225135],
        [-0.9382051 , -0.54304815],
        [ 1.72725924, -1.18582677],
        [ 1.77736657,  1.51157598],
        [ 1.89969252,  0.83444483],
        [-0.58723065, -1.97171753]]),
 (10, 2))

In [5]:
y ,y.shape

(array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0]), (10,))

In [6]:
X = torch.tensor(X,dtype = torch.float32)
y = torch.tensor(y,dtype = torch.long)

In [7]:
from torch.utils.data import Dataset ,DataLoader

In [8]:
class CustomDataset(Dataset):
    def __init__(self , features, label):
        self.features = features
        self.label = label

    def __len__(self):
        return self.features.shape[0] #sample size

    def __getitem__(self,index):
        return self.features[index] , self.label[index]

In [9]:
dataset = CustomDataset(X,y)

In [10]:
len(dataset)

10

In [11]:
dataset[2]

(tensor([-2.8954,  1.9769]), tensor(0))

In [12]:
dataloader = DataLoader(dataset,batch_size=2,shuffle=True)

other parameters of DataLoader:

* num_workers -> num of worker processes used to load data in parallel
* pin_memory -> makes data transfoer from CPU to GPU faster
* drop_last -> last match's sample frequency less than the rest , wanna keep the last batch or not ?
* collate_fn -> a callable that processes a list of samples into a batch
* sampler -> defines strategy for drawing samples


In [13]:
for batch_features , batch_labels in dataloader :
    print(batch_features)
    print(batch_labels)
    print("-"*50)

tensor([[ 1.7774,  1.5116],
        [ 1.7273, -1.1858]])
tensor([1, 1])
--------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-1.1402, -0.8388]])
tensor([0, 0])
--------------------------------------------------
tensor([[-0.7206, -0.9606],
        [-1.9629, -0.9923]])
tensor([0, 0])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [ 1.8997,  0.8344]])
tensor([1, 1])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [ 1.0683, -0.9701]])
tensor([0, 1])
--------------------------------------------------


## Example Usecase 

In [14]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [15]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [16]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [17]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [18]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [19]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [21]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [22]:
from torch.utils.data import Dataset , DataLoader

class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return self.features.shape[0]
    def __getitem__(self,index):
        return self.features[index],self.labels[index]

In [23]:
train_dataset = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

In [24]:
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size = 32 ,shuffle = True)

In [25]:
import torch.nn as nn

class MySimpleNN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()
    def forward(self,features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out 

In [26]:
lr = 0.1
epochs = 25

In [27]:
model = MySimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters() , lr = lr)
loss_func = nn.BCELoss()

In [28]:
for epoch in range(epochs):
    for batch_features , batch_labels in train_loader :
        y_pred = model(batch_features)
        loss = loss_func(y_pred,batch_labels.view(-1,1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.14348508417606354
Epoch: 2, Loss: 0.18948017060756683
Epoch: 3, Loss: 0.1429004818201065
Epoch: 4, Loss: 0.20901231467723846
Epoch: 5, Loss: 0.2300797402858734
Epoch: 6, Loss: 0.06535530835390091
Epoch: 7, Loss: 0.05088528245687485
Epoch: 8, Loss: 0.014603711664676666
Epoch: 9, Loss: 0.24706201255321503
Epoch: 10, Loss: 0.04190140217542648
Epoch: 11, Loss: 0.01022075954824686
Epoch: 12, Loss: 0.02266782522201538
Epoch: 13, Loss: 0.05458075553178787
Epoch: 14, Loss: 0.045188844203948975
Epoch: 15, Loss: 0.07889483124017715
Epoch: 16, Loss: 0.08329075574874878
Epoch: 17, Loss: 0.4203121066093445
Epoch: 18, Loss: 0.003264226485043764
Epoch: 19, Loss: 0.027679216116666794
Epoch: 20, Loss: 0.007427800912410021
Epoch: 21, Loss: 0.0029821989592164755
Epoch: 22, Loss: 0.06568238884210587
Epoch: 23, Loss: 0.022332560271024704
Epoch: 24, Loss: 0.13561400771141052
Epoch: 25, Loss: 0.04903260990977287


In [29]:
model.eval() #set model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features , batch_labels in test_loader:
        y_pred = model(batch_features)
        y_pred = (y_pred>0.9).float() #convert prob into binary pred
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

overall_acc = sum(accuracy_list) / len(accuracy_list)
print(f"Accuracy:{overall_acc:.4f}")

Accuracy:0.9253
